# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinod5566773/flyrank-capstone/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

My project uses one row per content item in the aggregated feature table. The model uses traffic, position, query, and engagement signals from two consecutive 30-day windows.

The main features are `imp_prev30`, `imp_last30`, `clk_last30`, `pos_last30`, `visible_queries`, `rare_share`, `anon_share`, and `top_query_share`. `kept_impressions` is included in the later model comparison.

The target is `is_declining`, defined as `imp_last30 < 0.8 * imp_prev30`. The identifiers `client_hash_id` and `content_hash_id` are retained for grouping and validation, not as predictive features. I use the existing feature table rather than rebuilding the large warehouse dataset.


In [1]:
numeric_features = [
    "imp_prev30",
    "imp_last30",
    "clk_last30",
    "pos_last30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "kept_impressions",
]

label_field = "is_declining"

identifier_fields = [
    "client_hash_id",
    "content_hash_id",
]

print("Documented numeric features:", len(numeric_features))
print("Target:", label_field)
print("Identifiers excluded from predictive inputs:", identifier_fields)

Documented numeric features: 9
Target: is_declining
Identifiers excluded from predictive inputs: ['client_hash_id', 'content_hash_id']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### 2. Feature notes: meaning, missing values, and availability

The model uses aggregated performance signals from two consecutive 30-day windows. `imp_prev30` represents impressions in the previous window, while `imp_last30` represents impressions in the recent window. `clk_last30` and `pos_last30` describe recent clicks and position. `visible_queries`, `rare_share`, `anon_share`, and `top_query_share` describe query visibility and the distribution of query activity. `kept_impressions` is used in the later model comparison.

The target `is_declining` equals 1 when recent impressions are less than 80% of previous-window impressions. The previous-window inclusion threshold is 100 impressions.

These are aggregated observational features. Their availability must be checked against the intended prediction time. Because the target is defined using `imp_last30`, including `imp_last30` as a model feature creates a direct target-rule leakage risk. A stricter future-prediction model should use features available before the future target window begins.

The analysis must not assume that missing values mean zero unless the data preparation explicitly applies that treatment. Client and content hashes are identifiers for grouping or validation, not predictive features.


In [2]:
feature_notes = {
    "imp_prev30": "Impressions in the previous 30-day window",
    "imp_last30": "Impressions in the recent 30-day window; overlaps with target rule",
    "clk_last30": "Clicks in the recent 30-day window",
    "pos_last30": "Position signal in the recent 30-day window",
    "visible_queries": "Count of visible queries",
    "rare_share": "Share of rare queries",
    "anon_share": "Share of anonymous queries",
    "top_query_share": "Share associated with top queries",
    "kept_impressions": "Kept impressions used in later model comparison",
}

for feature, meaning in feature_notes.items():
    print(f"{feature}: {meaning}")

imp_prev30: Impressions in the previous 30-day window
imp_last30: Impressions in the recent 30-day window; overlaps with target rule
clk_last30: Clicks in the recent 30-day window
pos_last30: Position signal in the recent 30-day window
visible_queries: Count of visible queries
rare_share: Share of rare queries
anon_share: Share of anonymous queries
top_query_share: Share associated with top queries
kept_impressions: Kept impressions used in later model comparison


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

### 3. The leakage hunt

The target is defined as `imp_last30 < 0.8 * imp_prev30`. Therefore, `imp_last30` directly participates in calculating the label. Using it as a predictive feature would allow the model to learn the target rule rather than independently predict a future decline.

I exclude `imp_last30` from the strict leakage-safe feature list. I also exclude `is_declining`, any target-derived fields, identifiers, post-decision outcomes, and measurements from future windows that would not be available at prediction time.

The existing capstone results are useful for comparing validation approaches, but they should not be presented as proof of future prediction performance without a time-separated target and features. This is an important limitation of the current task framing.


In [3]:
label_field = "is_declining"

identifier_fields = [
    "client_hash_id",
    "content_hash_id",
]

target_derived_fields = [
    "is_declining",
]

future_or_post_decision_fields = [
    "future-window measurements",
    "post-decision outcomes",
]

strict_feature_candidates = [
    "imp_prev30",
    "clk_last30",
    "pos_last30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share",
    "kept_impressions",
]

# imp_last30 is intentionally excluded because it defines the target.
assert "imp_last30" not in strict_feature_candidates
assert label_field not in strict_feature_candidates
assert not any(
    field in strict_feature_candidates
    for field in identifier_fields
)

print("Strict feature candidates:", strict_feature_candidates)
print("Target excluded:", label_field not in strict_feature_candidates)
print("Identifier fields excluded:", identifier_fields)
print("Recent impressions excluded due to target-rule overlap: True")

Strict feature candidates: ['imp_prev30', 'clk_last30', 'pos_last30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'kept_impressions']
Target excluded: True
Identifier fields excluded: ['client_hash_id', 'content_hash_id']
Recent impressions excluded due to target-rule overlap: True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### 4. What I excluded and why

* `client_hash_id` — identifies the client group and is used for grouping or validation, not as a predictive feature.
* `content_hash_id` — identifies a content item and is not an independent performance signal.
* `is_declining` — the target itself; it must not be an input.
* `imp_last30` — directly participates in the target rule, so it is excluded from the strict leakage-safe feature candidates.
* Future-window measurements — unavailable at the time the prediction would be made.
* Post-decision outcomes — would introduce information from after the decision.
* Any target-derived fields — would reveal the label rather than independently predict it.

The purpose of these exclusions is to make the analysis more honest and reduce leakage. Removing a field from a candidate list does not by itself establish that a model is leakage-free; the feature timeline and validation design must also be checked.


In [4]:
excluded_fields = {
    "client_hash_id": "Identifier for grouping or validation",
    "content_hash_id": "Content identifier, not a predictive signal",
    "is_declining": "Target itself",
    "imp_last30": "Directly participates in the target rule",
    "future-window measurements": "Unavailable at prediction time",
    "post-decision outcomes": "Information from after the decision",
}

for field, reason in excluded_fields.items():
    print(f"{field}: {reason}")

print("\nExclusion policy documented.")

client_hash_id: Identifier for grouping or validation
content_hash_id: Content identifier, not a predictive signal
is_declining: Target itself
imp_last30: Directly participates in the target rule
future-window measurements: Unavailable at prediction time
post-decision outcomes: Information from after the decision

Exclusion policy documented.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.